# 02 — Per-site K_d retrieval and bootstrap

This is the **main computation** of the paper. It runs the per-site
K_d sweep at Apollo 15 and 17 and the non-parametric bootstrap.

## Two ways to use this notebook

**FAST path (~5 min)**: Run only **Step 1** below. This regenerates the
canonical `output/kd_retrieval_results.json` with K_d*, bootstrap, Q_b
sensitivity, and joint (K_d, H) fit. That is enough to feed all
headline figures in notebooks 03 + 04.

**FULL path (~60 min)**: Also run **Step 2** to regenerate the
auxiliary sensitivity-sweep JSONs that feed Table 3 (per-component
error budget). These are independent computations and can be skipped
— their canonical outputs ship with the repository under `output/`.

Each script writes its result to a JSON file in `output/` and is
idempotent (re-running overwrites).

In [ ]:
import sys, pathlib, subprocess, json, time
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

## Step 1 — Phase A: central retrieval (~5 min)

Runs `scripts/pipeline/retrieve_kd.py` which:
1. Loads the bundled HFE record
2. Sweeps K_d on a fine grid at each site
3. Identifies the RMSE minimum (K_d*)
4. Runs 1500-resample bootstrap with +-2.5 cm depth jitter
5. Computes K_d / Q_b degeneracy and joint (K_d, H) fit
6. Writes canonical JSON + Figs 7 (bootstrap), 10 (robustness)

In [ ]:
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'pipeline' / 'retrieve_kd.py')],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError(f'retrieve_kd failed (exit {result.returncode})')
print(f'\nPhase A complete in {(time.time()-t0)/60:.1f} min.')

### Verify the headline result

In [ ]:
d = json.loads((ROOT / 'output' / 'kd_retrieval_results.json').read_text())
for site in ('A15', 'A17'):
    s = d[site]; b = s['bootstrap']
    print(f'  {site}:')
    print(f'    K_d*        = {s["kd_star"]*1e3:.3f} mW m^-1 K^-1')
    print(f'    RMSE at K_d*= {s["rmse_star"]:.3f} K')
    print(f'    bootstrap median = {b["median"]*1e3:.2f} mW m^-1 K^-1')
    print(f'    95% CI      = [{b["ci_lo"]*1e3:.2f}, {b["ci_hi"]*1e3:.2f}] mW m^-1 K^-1')
    print()
print('Expected (matches paper):')
print('  A15  K_d* = 4.58 mW m^-1 K^-1, CI [4.12, 7.45]')
print('  A17  K_d* = 8.12 mW m^-1 K^-1, CI [6.85, 9.04]')

## Step 2 — OPTIONAL: auxiliary sensitivity sweeps (~50 min)

These regenerate the JSONs that feed Table 3 (per-component error
budget). Each one runs additional K_d sweeps under perturbed inputs.

**Skip this whole step if you only want the headline figures.** The
canonical outputs are already in `output/`.

| Script | Produces | ~Time |
|---|---|---|
| `compute_borestem_sensitivity.py` | `borestem_sensitivity.json` (z_b cut sweep) | 5 min |
| `compute_stability_threshold_sensitivity.py` | `stability_threshold_sensitivity.json` | 5 min |
| `compute_surface_bias_test.py` | `surface_bias_test.json` (Bond albedo perturbation) | 8 min |
| `compute_uniform_kd_sensitivity.py` | `uniform_kd_test.json` (fine 41-point sweep) | 12 min |
| `compute_headline_rmse.py` | `headline_rmse.json` (Table 2 inputs) | 8 min |
| `compute_model_selection.py` | `model_selection.json` (AICc, Table 4 inputs) | 10 min |

Set `RUN_AUXILIARY = True` to execute. Default is False so the notebook
runs end-to-end in ~5 minutes.

In [ ]:
RUN_AUXILIARY = False    # change to True to regenerate all auxiliary JSONs

if RUN_AUXILIARY:
    scripts = [
        'compute_borestem_sensitivity.py',
        'compute_stability_threshold_sensitivity.py',
        'compute_surface_bias_test.py',
        'compute_uniform_kd_sensitivity.py',
        'compute_headline_rmse.py',
        'compute_model_selection.py',
        'compute_error_budget.py',
    ]
    for script in scripts:
        t0 = time.time()
        print(f'\n=== {script} ===')
        r = subprocess.run(
            [sys.executable, str(ROOT / 'scripts' / 'pipeline' / script)],
            cwd=str(ROOT), capture_output=True, text=True,
        )
        dt = (time.time() - t0) / 60.0
        if r.returncode != 0:
            print(f'  FAILED after {dt:.1f} min:', r.stderr[-300:])
        else:
            print(f'  ok in {dt:.1f} min')
else:
    print('Auxiliary sweeps SKIPPED (RUN_AUXILIARY = False).')
    print('Canonical JSONs are already in output/. Set RUN_AUXILIARY = True')
    print('above and re-run this cell to regenerate them (~50 min total).')

## Step 3 — OPTIONAL: Bayesian MCMC cross-check (~5 min)

Independent cross-check of the contrast direction (not magnitude).
Not required for any headline figure. Skip if you don't need the
Bayesian discussion section.

In [ ]:
RUN_MCMC = False    # change to True to run the MCMC cross-check

if RUN_MCMC:
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'pipeline' / 'bayesian_crosscheck.py')],
        cwd=str(ROOT), capture_output=True, text=True,
    )
    dt = (time.time() - t0) / 60.0
    if r.returncode != 0:
        print(f'Phase B MCMC failed after {dt:.1f} min:', r.stderr[-500:])
    else:
        print(f'Phase B MCMC complete in {dt:.1f} min.')
else:
    print('MCMC cross-check SKIPPED (RUN_MCMC = False).')

---

**Next**: open `03_results.ipynb` to render Figs 5-9 and Tables 2-3.